# 13 · Candidate target prioritisation (translational layer)
For genes with **no measured perturbation** we predict the response with an ensemble of models, summarise effect size, add external evidence (DepMap essentiality, cancer-gene lists, druggability, disease association), and rank. Uncertainty = disagreement across models.

⚠️ **Output is a ranked list of computational hypotheses, not validated drug targets.** Essentiality in K562 does not imply a therapeutic window; predictions must be validated experimentally.

In [ ]:
# ---- Bootstrap: Colab (Google Drive) or local ----
import os, sys, subprocess
REPO_URL = "https://github.com/<your-username>/PerturbAI.git"   # <-- edit once after you create the repo
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    if not os.path.exists("/content/PerturbAI"):
        subprocess.run(["git", "clone", REPO_URL, "/content/PerturbAI"], check=True)
    os.chdir("/content/PerturbAI")
    subprocess.run(["git", "pull", "-q"], check=False)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements-colab.txt"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", "."], check=True)
else:
    try:
        import perturbai  # installed with `pip install -e .`
    except ImportError:
        sys.path.insert(0, os.path.abspath("../src"))  # fallback when running from notebooks/
from perturbai.paths import get_paths
P = get_paths()
print("Data root:", P.root)

In [ ]:
import numpy as np, pandas as pd, scipy.sparse as sp, matplotlib.pyplot as plt
from perturbai.paths import artifact
from perturbai.utils import set_seed, load_config
CFG = load_config("data_config.yaml"); MCFG = load_config("model_config.yaml")
set_seed(CFG["seed"])
SYNTHETIC = False        # True = 1-minute smoke test on simulated data (no download)
DS = "replogle_k562_essential"
INFO = CFG["datasets"][DS]
PERT, CTRL, BATCH = INFO["perturbation_col"], INFO["control_label"], INFO["batch_col"]
TAG = DS + ("_synthetic" if SYNTHETIC else "")   # artifact prefix

In [ ]:
import joblib
from perturbai.pipeline import load_task
from perturbai.evaluation.metrics import rowwise_spearman
from perturbai.models.baselines import RidgeModel, KNNModel, RandomForestModel
from perturbai.translational.prioritize import prioritize, load_depmap_gene_effect
from scipy.stats import spearmanr
T = load_task(P, TAG, ("coexp_",)); Mc = MCFG["models"]
ens = {"ridge": RidgeModel(**Mc["ridge"]), "knn": KNNModel(**Mc["knn"]),
       "rf": RandomForestModel(Mc["rf"]["n_estimators"], Mc["rf"]["n_pca_targets"], Mc["rf"]["min_samples_leaf"])}
for m in ens.values(): m.fit(T["X"], T["Y"])
cand_genes = [g for g in T["feats"].index if g not in set(T["index"])]
Xc = T["feats"].loc[cand_genes].values
preds = np.stack([m.predict(Xc) for m in ens.values()])      # models x genes x outputs
mean_pred = preds.mean(0); unc = pd.Series(preds.std(0).mean(1), index=cand_genes)
pred_delta = pd.DataFrame(mean_pred, index=cand_genes, columns=T["genes"])
print("candidate genes (never perturbed in this subset):", len(cand_genes))

In [ ]:
# Sanity: does predicted effect SIZE track real effect size on measured perturbations? (in-sample for training genes, so interpret carefully)
real = np.linalg.norm(T["Y"], axis=1)
inS = np.linalg.norm(np.stack([m.predict(T["X"]) for m in ens.values()]).mean(0), axis=1)
print("spearman(real, predicted effect size), in-sample:", round(spearmanr(real, inS)[0], 3))

In [ ]:
# External evidence (optional files you download yourself; see docs/DATA_SOURCES.md)
ev = None
dm = P.raw / "depmap" / "CRISPRGeneEffect.csv"
if dm.exists():
    mp = P.raw / "depmap" / "Model.csv"
    ev = load_depmap_gene_effect(dm, mp if mp.exists() else None, "K562")
    print("DepMap evidence loaded:", ev.shape)
for name, f in {"is_cancer_gene": "cancer_genes.csv", "druggability": "druggability.csv", "disease_assoc": "disease_assoc.csv"}.items():
    fp = P.raw / "evidence" / f      # CSV with columns: gene, value
    if fp.exists():
        v = pd.read_csv(fp).set_index("gene").iloc[:, 0].rename(name)
        ev = v.to_frame() if ev is None else ev.join(v, how="outer")
print("evidence columns:", None if ev is None else list(ev.columns))

In [ ]:
rank = prioritize(pred_delta, ev, uncertainty=unc)
rank.to_csv(P.results / artifact("priority", TAG))
display(rank.head(25)[[c for c in rank.columns if c in ["l2_norm", "n_genes_changed", "composite", "composite_adj", "uncertainty", "depmap_mean_effect", "druggability"]]].round(3))
print(rank.caveat.iloc[0])

## Report guidance
- Describe the evidence sources, versions and access dates; state the weighting is a modelling choice, not ground truth.
- Show 3–5 top candidates with: predicted response summary, evidence table, uncertainty, and why each might fail.
- Never write "drug target"; write "candidate for follow-up".